# Demo — Stewardship on the Ambiguous Band

Identity resolution auto-merges the obvious duplicates and rejects the obvious non-matches —
but the **ambiguous band** in between needs human judgment. This walkthrough shows the match
bands, compares an `auto-merge` pair against an `ambiguous` pair, and demonstrates applying a
single steward adjudication with `apply_adjudications()`. The exercise asks you to adjudicate
all six ambiguous pairs and document your survivorship rule.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import GovernedCatalog, apply_adjudications

gc = GovernedCatalog("trailhead.db")
m = gc.resolve_customers()
print("auto-merge:", int((m.pairs.band == "auto-merge").sum()),
      "| ambiguous (review queue):", len(m.ambiguous),
      "| rejected:", int((m.pairs.band == "reject").sum()))

## Compare `auto-merge` vs. `ambiguous` candidate pairs
Pairs scoring `>= 0.85` share an exact email and normalized name, so the pipeline merges them
automatically. Pairs in the `0.66–0.84` band land in `m.ambiguous` for steward review:

In [ ]:
print("--- Auto-merged pairs (score >= 0.85) ---")
display(m.pairs[m.pairs.band == "auto-merge"][["id_a", "name_a", "email_a", "id_b", "name_b", "email_b", "score", "band"]])

print("\n--- First ambiguous pair in the steward queue (row 0) ---")
m.ambiguous.iloc[[0]][["id_a", "name_a", "email_a", "id_b", "name_b", "email_b", "score"]]

## Adjudicate one pair and inspect survivorship
Row `0` has the same customer name and email local-part, with only a swapped email domain —
strong signal that one person registered twice. Pass `{0: "accept"}` to `apply_adjudications()`
to see how the survivorship rule (`lowest customer_id wins`) selects the golden record:

In [ ]:
sample_outcome = apply_adjudications(m, decisions={0: "accept"})
print("Accepted merges:", sample_outcome["accepted_merges"], "of", sample_outcome["ambiguous_pairs"])
sample_outcome["merges"]

**Why human judgment matters:** accepting a true duplicate consolidates scattered loyalty and
marketing profiles into a single golden record (`golden_id`), whereas falsely merging two
different people who merely share a last name would leak one customer's PII to another.

**Takeaway / your turn:** in the exercise you'll inspect and adjudicate all six ambiguous pairs,
apply your decisions with `apply_adjudications()`, and write the stewardship decision log and
survivorship rule.